# Number Partitioning

Here we show how to solve the number partitioning problem using OpenJij, [JijModeling](https://jij-inc-jijmodeling-tutorials-en.readthedocs-hosted.com/en/latest/introduction.html), and ommx-openjij-adapter.
This problem is also mentioned in 2.1. Number Partitioning in [Lucas, 2014, "Ising formulations of many NP problems"](https://doi.org/10.3389/fphy.2014.00005).

## Overview of the Number Partitioning Problem
Number partitioning is the problem of dividing a given set of numbers into two subsets such that the sum of the numbers is equal.

### Example

Let us have a set of numbers $A=\{1, 2, 3, 4\}$.
It is easy to divide this set into equal sums: $\{1, 4\}, \{2, 3\}$ and the sum of each subset is 5.
Thus, when the size of the set is small, the answer is relatively easy to obtain.
When the problem is large, however, it is not immediately solvable.
Here, we solve this problem using annealing.

## Mathematical Model
First, let us model the Hamiltonian of this problem.
Let $A$ be the set to be partitioned and $a_i (i = \{0,1,\dots,N-1\})$ be its elements.
Here $N$ is the number of elements in this set.
We consider dividing $A$ into two sets $A_0$ and $A_1$.
Let $x_i$ be a variable whose $i$th element of $A$ is 0 when it is contained in the set $A_0$ and 1 when it is contained in $A_1$.
Using this variable $x_i$, the total value of the numbers in $A_0$ is written as $\sum_i a_i (1-x_i)$ and $\sum_i a_i x_i$ for $A_1$.
As we find a solution that satisfies the constraint that the sum of the numbers contained in $A_0$ and $A_1$ be equal, this can be expressed as:

$$\sum_i a_i (1-x_i)=\sum_i a_i x_i$$

The problem is to find $x_i$ that satisfies the constraint.
By transforming this expression, we can write $\sum_i a_i (2-x_i)=0$, and by using the Penalty method and squaring this constraint, the Hamiltonian for the number-splitting problem is:

$$H=\left( \sum_{i=0}^{N-1} a_i (2-x_i)\right)^2$$

## Modeling with JijModeling
Next, we show how to implement the above mathematical model using JijModeling. We first define the variables and parameters used in the model.

In [1]:
import jijmodeling as jm

problem = jm.Problem("Number Partition")

a = problem.Float("a", ndim=1)
N = problem.DependentVar("N", a.len_at(0))
x = problem.BinaryVar("x", shape=(N, ))

### Constraint
The constraint in equation (1) can be implemented as follows.

In [2]:
problem += problem.Constraint("equal", (jm.sum(N, lambda i: a[i]*(1-x[i])) == jm.sum(N, lambda i: a[i]*x[i])))

Let us check the implementation in the Jupyter Notebook.

In [3]:
problem

Problem(name="Number Partition", sense=MINIMIZE, objective=0, constraints={equal: [Constraint(name="equal", sense=EQUAL, left=sum(N.map(lambda (i: natural): a[i] * (1 - x[i]))), right=sum(N.map(lambda (i: natural): a[i] * x[i])), shape=Scalar(Float)),],})

## Creating an Instance
As an example, let us solve an easy problem; let us consider the problem of dividing numbers from 1 to 40.
When dividing consecutive numbers from $N_i$ to $N_f$ and keeping the total number of consecutive numbers even, there are several patterns of division.
However, the total value of the divided set is:

$$\mathrm{total\ value} = \frac{(N_{i} + N_{f})(N_{f} - N_{i} + 1)}{4}$$

In this case, the total value is expected to be 410.
Let us confirm this.

In [4]:
import numpy as np

inst_N = 40
instance_data = {"a": np.arange(1, inst_N+1)}

## Running Optimization with OpenJij

Let us solve the optimization problem using OpenJij's simulated annealing.

In [5]:
from ommx_openjij_adapter import OMMXOpenJijSAAdapter

instance = problem.eval(instance_data)

adapter = OMMXOpenJijSAAdapter(instance)
best_sample = adapter.sample(instance, num_reads=10).best_feasible_unrelaxed

## Visualizing the Solution

Here, we separate the indices classified into $A_0$ and $A_1$ in $A$, and compute their sums.

In [6]:
# decode a result to JijModeling sampleset
# get the indices of x == 1
df = best_sample.decision_variables_df
class_0_indices = [row['subscripts'][0] for _, row in df.iterrows() if row['value'] == 0.0]
class_1_indices = [row['subscripts'][0] for _, row in df.iterrows() if row['value'] == 1.0]

class_0 = instance_data["a"][class_0_indices]
class_1 = instance_data["a"][class_1_indices]

print(f"class 0 : {class_0} , total value = {np.sum(class_0)}")
print(f"class 1 : {class_1} , total value = {np.sum(class_1)}")

class 0 : [ 5  7 10 12 13 15 17 18 19 20 22 24 25 26 30 33 37 38 39] , total value = 410
class 1 : [ 1  2  3  4  6  8  9 11 14 16 21 23 27 28 29 31 32 34 35 36 40] , total value = 410


As expected, both total values are 410.
Above, we dealt with a problem whose solution is known because it is a consecutive number.
We recommend you try more complex problems, such as generating numbers randomly.